In [12]:
# 0. Establish API key

import os

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

#OPENAI_API_KEY

#ANTHROPIC_API_KEY

if GEMINI_API_KEY is None:
    raise RuntimeError("GEMINI_API_KEY not found in environment.")

In [13]:
# 1. Run config specs

run_config = {
  "run_id": None,

  "feature_name": "DtExp",

  "model_provider": "gemini",
  "model_name": "gemini-2.5-flash",  # <-- set this correctly

  "input_csv_path": "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/input/ntb_input.csv",     # <-- set
  "cri_json_path": "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/cris/DtExp.json",     # <-- set

  "data": {
      "max_tokens": 10,       # None = all
      "start_index": 0,
      "shuffle": False,
      "random_seed": 0
  },

  "permutation": {
    "type": "full",            # full | ablation | allation
    "component": None
  },

  "prompt": {
    "version": "v1",
    "system_template_hash": None,
    "assembled_prompt_hash": None
  },

  "inference": {
    "temperature": 0.0,
    "max_tokens": 512,
    "seed": None,
    "rate_limit_sleep_s": 1.0,
    "max_retries": 5
  },

  "io": {
    "output_dir": None,
    "jsonl_output_path": None,
    "csv_output_path": None
  },

  "notes": ""
}

In [14]:
#from google import genai

#client = genai.Client(api_key=GEMINI_API_KEY)

#response = client.models.generate_content(
#    model=run_config["model_name"],
#    contents="Respond with exactly the word: ALIVE"
#)

#print(response.text)

In [15]:
# 2. Auto-generate run_id, create output directory, and write config to disk

import json
from datetime import datetime
import uuid
import os

# ---- 1. Generate run_id ----
timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
short_uuid = str(uuid.uuid4())[:8]
run_id = f"{timestamp}_{short_uuid}"

run_config["run_id"] = run_id

# ---- 2. Define output directory structure ----
base_output_dir = "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/runs/Gemini/DtExp"
run_output_dir = os.path.join(base_output_dir, run_id)

os.makedirs(run_output_dir, exist_ok=True)

run_config["io"]["output_dir"] = run_output_dir

# IMPORTANT:
# We will set jsonl_output_path and csv_output_path
# AFTER we know which tokens are being processed.

run_config["io"]["jsonl_output_path"] = None
run_config["io"]["csv_output_path"] = None

# ---- 3. Write config to disk immediately ----
config_path = os.path.join(run_output_dir, "run_config.json")

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

print(f"Run initialized: {run_id}")
print(f"Config written to: {config_path}")

Run initialized: 20260224T210644Z_63b53001
Config written to: /Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/runs/20260224T210644Z_63b53001/run_config.json


/var/folders/22/4lgkjkkn2vvb2m8ffgsjl7gh0000gn/T/ipykernel_91233/1365625495.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")


In [16]:
# 3. Load CRI json file and implement permutation (full/ablation/allation)

import json
from copy import deepcopy

CRI_COMPONENTS = [
    "intuition",
    "set_to_1_if",
    "set_to_0_if",
    "edge_cases",
    "examples_positive",
    "examples_negative",
]

def load_cri(cri_json_path: str) -> dict:
    with open(cri_json_path, "r", encoding="utf-8") as f:
        cri = json.load(f)
    return cri

def apply_permutation(cri: dict, permutation: dict) -> dict:
    """
    Returns a CRI dict containing ONLY the active components for this run.
    - full: all components kept
    - ablation: drop exactly one named component
    - allation: keep exactly one named component
    """
    ptype = permutation["type"]
    component = permutation.get("component")

    # Defensive copy
    cri_active = deepcopy(cri)

    if ptype == "full":
        # Ensure we only keep the known components (avoids stray metadata leaking into prompt)
        return {k: cri_active.get(k) for k in CRI_COMPONENTS if k in cri_active}

    if component not in CRI_COMPONENTS:
        raise ValueError(f"Invalid component '{component}'. Must be one of: {CRI_COMPONENTS}")

    if ptype == "ablation":
        return {k: cri_active.get(k) for k in CRI_COMPONENTS if k != component and k in cri_active}

    if ptype == "allation":
        return {component: cri_active.get(component)}

    raise ValueError(f"Invalid permutation type '{ptype}'. Must be: full | ablation | allation")

# ---- Fill these in for your current run ----
run_config["cri_json_path"] = "/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/cris/DtExp.json"   # <-- set this
# run_config["permutation"] = {"type": "ablation", "component": "edge_cases"}  # example
# run_config["permutation"] = {"type": "allation", "component": "set_to_1_if"} # example

cri = load_cri(run_config["cri_json_path"])
cri_active = apply_permutation(cri, run_config["permutation"])

print("Active CRI components:", list(cri_active.keys()))

Active CRI components: ['intuition', 'set_to_1_if', 'set_to_0_if', 'edge_cases', 'examples_positive', 'examples_negative']


In [17]:
# 4. Assemble a deterministic prompt string + write its hash into run_config

import hashlib

def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

PROMPT_TEMPLATE_V1 = """\
You are a trained linguist performing binary semantic feature annotation.

TASK:
Decide whether the TARGET GRAM instantiates the feature: {feature_name}.

EVIDENCE RULES:
- Use only the TARGET GRAM and the FULL TEXT provided.
- Follow the Constrained Reasoning Instructions (CRIs) exactly as written.
- Do not assume missing context or world knowledge beyond what is in the FULL TEXT.

META-INSTRUCTIONS (how to use the CRIs):
{meta_block}

CRIs:
{cri_block}

INPUT:
TokenID: {TokenID}
Target: {Target}
FullText: {FullText}

OUTPUT (STRICT JSON ONLY; no extra text):
{{
  "TokenID": "{TokenID}",
  "annotation": 0 or 1,
  "confidence": "high" or "medium" or "low",
  "explanation": "one sentence"
}}
"""

def render_meta_block(cri_active: dict) -> str:
    lines = []

    # Always present
    lines.append("Treat each CRI component as a different kind of evidence. Follow them as written.")

    # Only if set_to_1_if exists
    if "set_to_1_if" in cri_active:
        lines.append("Assign 1 if at least one 'set_to_1_if' condition applies.")

    # Only if set_to_0_if exists
    if "set_to_0_if" in cri_active:
        lines.append(
            "If any 'set_to_0_if' condition applies, assign 0 (this vetoes 'set_to_1_if'), "
            "unless an 'edge_cases' instruction explicitly overrides."
        )

    return "- " + "\n- ".join(lines)

def render_cri_block(cri_active: dict) -> str:
    parts = []
    for key in CRI_COMPONENTS:
        if key in cri_active and cri_active[key] is not None:
            parts.append(f"[{key}]\n{cri_active[key]}")
    return "\n\n".join(parts)

def build_prompt(feature_name: str, cri_active: dict, token: dict) -> str:
    cri_block = render_cri_block(cri_active)
    meta_block = render_meta_block(cri_active)

    return PROMPT_TEMPLATE_V1.format(
        feature_name=feature_name,
        cri_block=cri_block,
        meta_block=meta_block,
        TokenID=token["TokenID"],
        Target=token["Target"],
        FullText=token["FullText"],
    )

# ---- store template hash in config ----
run_config["prompt"]["version"] = "v1"
run_config["prompt"]["system_template_hash"] = sha256_text(PROMPT_TEMPLATE_V1)

print("Template hash:", run_config["prompt"]["system_template_hash"])

Template hash: a162a52f2df71cfe965f20df69fa760eb7aaf1f55cd929401825f456846ab44a


In [18]:
# 5. Load input CSV, apply slice, set output filenames, and render one dry-run prompt

import pandas as pd
import os
import json
from datetime import datetime

# ---- Load CSV once ----
df = pd.read_csv(run_config["input_csv_path"])

required_columns = {"TokenID", "Target", "FullText"}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ---- Apply run_config data slicing deterministically ----
data_cfg = run_config["data"]

df_work = df
if data_cfg.get("shuffle", False):
    df_work = df_work.sample(frac=1, random_state=data_cfg.get("random_seed", 0)).reset_index(drop=True)

start = int(data_cfg.get("start_index", 0))
max_tokens = data_cfg.get("max_tokens", None)

df_slice = df_work.iloc[start:] if max_tokens is None else df_work.iloc[start:start + int(max_tokens)]
tokens = df_slice.to_dict(orient="records")

print(f"Total tokens in file: {len(df)}")
print(f"Tokens selected for this run: {len(tokens)}")

if not tokens:
    raise RuntimeError("No tokens selected for this run (tokens list is empty).")

# ---- Set output filenames: FEAT_FIRSTTOKEN_LASTTOKEN_YYYYMMDD ----
feat = run_config["feature_name"]
first_tok = str(tokens[0]["TokenID"])
last_tok = str(tokens[-1]["TokenID"])
date_str = datetime.now().strftime("%Y%m%d")

base = f"{feat}_{first_tok}_{last_tok}_{date_str}"

run_output_dir = run_config["io"]["output_dir"]
run_config["io"]["jsonl_output_path"] = os.path.join(run_output_dir, f"{base}.jsonl")
run_config["io"]["csv_output_path"] = os.path.join(run_output_dir, f"{base}.csv")

print("JSONL path:", run_config["io"]["jsonl_output_path"])
print("CSV path:  ", run_config["io"]["csv_output_path"])

# ---- Update config on disk so it matches the computed output paths ----
config_path = os.path.join(run_output_dir, "run_config.json")
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

# ---- Dry-run: build and inspect ONE prompt ----
first_token = tokens[0]
prompt0 = build_prompt(run_config["feature_name"], cri_active, first_token)

print("\n--- PROMPT (first token) ---\n")
print(prompt0[:2000])  # truncate so you don't flood the notebook
print("\n--- END PROMPT PREVIEW ---\n")

# store an assembled prompt hash for this first instance (useful debugging)
run_config["prompt"]["assembled_prompt_hash"] = sha256_text(prompt0)

Total tokens in file: 594
Tokens selected for this run: 50
JSONL path: /Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/runs/20260224T210644Z_63b53001/DtExp_40001018_40019027_20260224.jsonl
CSV path:   /Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/runs/20260224T210644Z_63b53001/DtExp_40001018_40019027_20260224.csv

--- PROMPT (first token) ---

You are a trained linguist performing binary semantic feature annotation.

TASK:
Decide whether the TARGET GRAM instantiates the feature: DtExp.

EVIDENCE RULES:
- Use only the TARGET GRAM and the FULL TEXT provided.
- Follow the Constrained Reasoning Instructions (CRIs) exactly as written.
- Do not assume missing context or world knowledge beyond what is in the FULL TEXT.

META-INSTRUCTIONS (how to use the CRIs):
- Treat each CRI component as a different kind of evidence. Follow them as written.
- Assi

In [19]:
# 6. Strict response parser + validator

import json
import re

_FENCE_RE = re.compile(r"^\s*```(?:json)?\s*\n([\s\S]*?)\n```\s*$", re.IGNORECASE)

def parse_model_response(raw_text: str) -> dict:
    """
    Parses model output and enforces strict JSON schema.
    Accepts either:
      - pure JSON
      - JSON wrapped in a single ```json ... ``` fence
    """
    text = raw_text.strip()

    m = _FENCE_RE.match(text)
    if m:
        text = m.group(1).strip()

    try:
        data = json.loads(text)
    except json.JSONDecodeError as e:
        raise ValueError(f"Model output is not valid JSON. JSONDecodeError: {e}")

    required_keys = {"TokenID", "annotation", "confidence", "explanation"}
    if set(data.keys()) != required_keys:
        raise ValueError(f"JSON keys mismatch. Expected {required_keys}, got {set(data.keys())}")

    if data["annotation"] not in (0, 1):
        raise ValueError("Annotation must be 0 or 1.")

    if data["confidence"] not in ("high", "medium", "low"):
        raise ValueError("Confidence must be 'high', 'medium', or 'low'.")

    if not isinstance(data["TokenID"], str):
        raise ValueError("TokenID must be a string.")

    if not isinstance(data["explanation"], str):
        raise ValueError("Explanation must be a string.")

    return data

In [24]:
# 7. Model call wrapper (Gemini + provider dispatcher)

import time
from google import genai
from google.genai import errors

# ---- Gemini client ----
client = genai.Client(api_key=GEMINI_API_KEY)

def call_gemini(prompt: str) -> str:
    max_retries = int(run_config["inference"].get("max_retries", 3))
    base_sleep = float(run_config["inference"].get("rate_limit_sleep_s", 1.0))

    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(
                model=run_config["model_name"],
                contents=prompt,
                config={
                    "temperature": run_config["inference"]["temperature"],
                    "max_output_tokens": run_config["inference"]["max_tokens"],
                },
            )

            if not response.text:
                candidate = response.candidates[0] if response.candidates else None
                finish_reason = getattr(candidate, "finish_reason", "unknown")
                raise ValueError(f"Empty response from model. finish_reason={finish_reason!r}, prompt_feedback={response.prompt_feedback!r}")
    
            return response.text

        except errors.ClientError as e:
            last_err = e
            # 429 = resource exhausted / rate limit / quota
            if getattr(e, "status_code", None) == 429 or "RESOURCE_EXHAUSTED" in str(e):
                sleep_s = max(base_sleep, 1.0) * (2 ** (attempt - 1))
                print(f"[429] Resource exhausted. Sleeping {sleep_s:.1f}s then retrying ({attempt}/{max_retries})...")
                time.sleep(sleep_s)
                continue
            raise

    raise RuntimeError(f"Gemini call failed after {max_retries} retries. Last error: {last_err}")

# ---- Provider dispatcher (future-proof) ----
def call_model(prompt: str) -> str:
    provider = run_config["model_provider"]

    if provider == "gemini":
        return call_gemini(prompt)

    # We will implement these in the next thread
    elif provider == "openai":
        raise NotImplementedError("OpenAI wrapper not yet implemented.")
    elif provider == "anthropic":
        raise NotImplementedError("Anthropic wrapper not yet implemented.")

    else:
        raise ValueError(f"Unknown model_provider: {provider}")

In [21]:
# 8. Single-token live test

#first_token = tokens[0]

#prompt0 = build_prompt(
#    run_config["feature_name"],
#    cri_active,
#    first_token
#)

#print("Sending prompt for TokenID:", first_token["TokenID"])

#raw_output = call_model(prompt0)

#print("\n--- RAW MODEL OUTPUT ---\n")
#print(raw_output)
#print("\n--- END RAW OUTPUT ---\n")

In [25]:
# 9. Batch annotation loop

import json
import time

output_path = run_config["io"]["jsonl_output_path"]

if output_path is None:
    raise RuntimeError("jsonl_output_path has not been set. Check Cell 5.")

print(f"Writing outputs to: {output_path}")
print(f"Processing {len(tokens)} tokens...")

processed = 0
parse_failures = 0

# IMPORTANT: use "w" so reruns don't silently append
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, "w", encoding="utf-8") as f:

    for token in tokens:
        token_id = token["TokenID"]
        prompt = build_prompt(run_config["feature_name"], cri_active, token)

        try:
            raw = call_model(prompt)
            parsed = parse_model_response(raw)

            # Add provenance so it travels with the data
            parsed["run_id"] = run_config["run_id"]
            parsed["model_provider"] = run_config["model_provider"]
            parsed["model_name"] = run_config["model_name"]
            parsed["feature_name"] = run_config["feature_name"]
            parsed["permutation_type"] = run_config["permutation"]["type"]
            parsed["permutation_component"] = run_config["permutation"]["component"]

            # Write clean JSON line
            f.write(json.dumps(parsed) + "\n")
            processed += 1

        except Exception as e:
            # Log structured error record instead of crashing
            error_record = {
                "TokenID": token_id,
                "annotation": None,
                "confidence": None,
                "explanation": None,
                "error": str(e)
            }

            f.write(json.dumps(error_record) + "\n")
            parse_failures += 1

        # small sleep to reduce burst pressure
        time.sleep(run_config["inference"]["rate_limit_sleep_s"])

print("Done.")
print("Processed:", processed)
print("Parse failures:", parse_failures)

Writing outputs to: /Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/runs/20260224T210644Z_63b53001/DtExp_40001018_40019027_20260224.jsonl
Processing 50 tokens...


FileNotFoundError: [Errno 2] No such file or directory: '/Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/runs/20260224T210644Z_63b53001/DtExp_40001018_40019027_20260224.jsonl'

In [23]:
# 10. Merge JSONL results back into CSV

import pandas as pd
import json

jsonl_path = run_config["io"]["jsonl_output_path"]
csv_output_path = run_config["io"]["csv_output_path"]

if jsonl_path is None or csv_output_path is None:
    raise RuntimeError("Output paths not set. Check Cell 5.")

# Reconstruct the exact input slice from tokens (no re-slicing logic)
df_slice = pd.DataFrame(tokens)

# Add run metadata to every row (redundant with JSONL, but very useful)
df_slice["run_id"] = run_config["run_id"]
df_slice["model_provider"] = run_config["model_provider"]
df_slice["model_name"] = run_config["model_name"]
df_slice["feature_name"] = run_config["feature_name"]
df_slice["permutation_type"] = run_config["permutation"]["type"]
df_slice["permutation_component"] = run_config["permutation"]["component"]

# Load JSONL results
records = []
with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

df_results = pd.DataFrame(records)

# If JSONL already includes these metadata fields, drop them to prevent *_x/*_y columns after merge
df_results = df_results.drop(
    columns=["run_id", "model_provider", "model_name", "feature_name",
             "permutation_type", "permutation_component"],
    errors="ignore"
)

# Merge on TokenID
df_merged = df_slice.merge(df_results, on="TokenID", how="left")

df_merged.to_csv(csv_output_path, index=False)

print("Merged CSV written to:", csv_output_path)

Merged CSV written to: /Users/glossophilia/Library/CloudStorage/GoogleDrive-rumilofaniel@gmail.com/My Drive/Diss/Project Files/p1_cri_pipeline/runs/20260224T210644Z_63b53001/DtExp_40001018_40019027_20260224.csv
